# fly-chess on Kaggle

Trains a fly on Kaggle's free GPU while the laptop does something else. As it stands it runs the control for the laptop's day-long fly: the same recipe (`flychess/train_cached.py`, the brain once over every position, then the readout fitted to the end) on the **shuffled** brain, same neurons and synapse strengths, random targets. That's the comparison that says whether the real wiring matters, after a lot more training than the 3-minute probes in the README.

1. New notebook, *File, Import notebook*, this file.
2. *Settings*: accelerator **GPU P100**, internet **on**. The code uses one GPU, and a P100 has more than twice a T4's memory bandwidth, which is what this brain needs.
3. The code: GitHub only has the first version. Either push the repo first, or zip the repo without `data/`, `checkpoints/` and `runs/` (keep `data/curriculum/*.parquet`, it's small), upload the zip as a private dataset, and attach it with *Add Input*. The first cell finds it.
4. *Save Version, Save & Run All (Commit)* and close the laptop. It stops itself well inside Kaggle's 12 hours.
5. To carry on in a new session, attach this notebook's output to the next version (*Add Input, Notebook Output*). It picks up where it stopped.
6. When it's done: the *Output* tab, download `checkpoints/` and `runs/` into your fly-chess folder.


In [ ]:
# the code: an attached dataset with the repo in it, or GitHub
import pathlib, shutil, subprocess
work = pathlib.Path('/kaggle/working/fly-chess')
src = next(pathlib.Path('/kaggle/input').glob('**/flychess/train_cached.py'), None)
if src and not work.exists():
    shutil.copytree(src.parent.parent, work, ignore=shutil.ignore_patterns('checkpoints', 'runs'))
elif not work.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/S1MS4/fly-chess.git', str(work)], check=True)
%cd /kaggle/working/fly-chess
!pip install -q python-chess msgpack zstandard
import tensorflow as tf; print(tf.__version__, tf.config.list_physical_devices('GPU'))

In [ ]:
# a previous session's output, if one is attached: the checkpoints and the cache carry on from there
prev = next((p for p in pathlib.Path('/kaggle/input').glob('**/checkpoints') if p.is_dir()), None)
if prev and not pathlib.Path('checkpoints').exists():
    shutil.copytree(prev, 'checkpoints')
    shutil.copytree(prev.parent / 'runs', 'runs', dirs_exist_ok=True)
    print('carrying on from', prev.parent)

In [ ]:
# the brain (570 MB download, about a minute to build) and the same 3 million positions as the laptop,
# so the held-out positions are the same ones (about 15 minutes)
!python scripts/download_connectome.py && python -m flychess.connectome --build
!python scripts/download_chessbench.py --files 6 && python -m flychess.train_dl --prepare

In [ ]:
# the shuffled brain, the day-long fly's recipe; leave out --shuffle-wiring for the real brain.
# --more-hours adds more ChessBench files after the first fit. Kaggle won't save an output over 20 GB, and
# the cache of the first 3 million positions is already 4 GB, so an hour of more files (about 5 GB) is the safe end.
!timeout 11h python -m flychess.train_cached --name dl-day-shuffled --shuffle-wiring --eye fair --read 9 10 11 12 --curriculum 0.1 --more-hours 1